In [34]:
%pip install openmeteo-requests
%pip install requests-cache retry-requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [35]:
import openmeteo_requests
import polars as pl
import requests_cache
from retry_requests import retry
from datetime import datetime, timedelta, timezone

In [36]:
# api config
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

url = "https://api.open-meteo.com/v1/forecast"

Functionise the below to make easier

In [37]:
# Call config
# Oxford
_latitude = 52.52
_longitude = 13.41

# The order of variables in hourly or daily is important to assign them correctly below
# Would be better to define a dictionary of the variables then pass in the names/order to use in later arguments. Then there is no risk of assigning values of one and mislabelling them.
var_list = [
    "temperature_2m", 
    "relative_humidity_2m", 
    "apparent_temperature", 
    "precipitation_probability", 
    "precipitation", "rain", 
    "showers", "snowfall", 
    "snow_depth", 
    "weather_code", 
    "cloud_cover", 
    "wind_speed_10m"
]

params = {
	"latitude": _latitude,
	"longitude": _longitude,
	"hourly": var_list,
}



In [ ]:
# api call
responses = openmeteo.weather_api(url, params=params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
hourly = response.Hourly()

# Setting up the datetime length to create the dataframe with
start = datetime.fromtimestamp(hourly.Time(), timezone.utc)
end = datetime.fromtimestamp(hourly.TimeEnd(), timezone.utc)
freq = timedelta(seconds = hourly.Interval())

# Dataframe base
# get the columns before populating with metrics
df_base = pl.select(
    lat = response.Latitude(),
    long = response.Longitude(),
    extract_date = datetime.fromtimestamp(hourly.Time(), timezone.utc).date(),
    extract_time = datetime.now().time(),
    fc_datetime = pl.datetime_range(start, end, freq, closed="left")
).with_columns(
    fc_date = pl.col('fc_datetime').dt.date(),
    fc_time = pl.col('fc_datetime').dt.time()
)

col_exprs = {var: hourly.Variables(var_list.index(var)).ValuesAsNumpy() for var in var_list}

df_metrics = pl.DataFrame(
    col_exprs
)

df_raw = pl.concat([df_base, df_metrics], how='horizontal')

